1. Create sample customers dataset, use following prompt:
"""
create sample dataset named customers. The customer dataset will have columns like customer_id, name, gender, address1, address2, city, state, country, zip, phone_1, phone_2, age. Generate around 1000 rows and store this dataset as csv file in volume brij_lakehouse.bronze.data.customers.csv
"""

In [0]:
# Create the customers view
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers_view AS
SELECT
  id AS customer_id,
  concat('Name_', id) AS name,
  CASE WHEN id % 2 = 0 THEN 'Male' ELSE 'Female' END AS gender,
  concat('Address1_', id) AS address1,
  concat('Address2_', id) AS address2,
  concat('City_', (id % 50) + 1) AS city,
  concat('State_', (id % 20) + 1) AS state,
  'Country_X' AS country,
  lpad(cast((10000 + id) as string), 5, '0') AS zip,
  concat('+1-555-', lpad(cast(id as string), 4, '0')) AS phone_1,
  concat('+1-556-', lpad(cast(id as string), 4, '0')) AS phone_2,
  18 + (id % 60) AS age
FROM
  (SELECT sequence(1, 10000) AS ids) 
  LATERAL VIEW explode(ids) AS id
""")

# Write the view to CSV in the volume
df = spark.table("customers_view")
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/brij_lakehouse/bronze/data/customers.csv")

print("Customer data written successfully to /Volumes/brij_lakehouse/bronze/data/customers.csv/")

2. create sample products datase, use following prompt: 

""" 
create sample dataset named products. The products dataset will have columns like product_id, name, price, color, weight, vendor_name. Generate around 10000 rows and store this dataset as csv file prodcuts.csv in volume brij_lakehouse.bronze.data 

"""

In [0]:
# Create the products view
spark.sql("""
CREATE OR REPLACE TEMP VIEW products_view AS
SELECT
  id AS product_id,
  concat('Product_', id) AS name,
  round(10 + (id % 500) * 1.5, 2) AS price,
  CASE WHEN id % 5 = 0 THEN 'Red'
       WHEN id % 5 = 1 THEN 'Blue'
       WHEN id % 5 = 2 THEN 'Green'
       WHEN id % 5 = 3 THEN 'Yellow'
       ELSE 'Black' END AS color,
  round(0.5 + (id % 100) * 0.1, 2) AS weight,
  concat('Vendor_', (id % 50) + 1) AS vendor_name
FROM
  (SELECT sequence(1, 10000) AS ids)
  LATERAL VIEW explode(ids) AS id
""")

# Write the view to CSV in the volume
df_products = spark.table("products_view")
df_products.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/brij_lakehouse/bronze/data/prodcuts.csv")

print("Products data written successfully to /Volumes/brij_lakehouse/bronze/data/prodcuts.csv")

%md
3. create sample orders dataset, use following prompt: 

""" 
create sample dataset named orders. The orders dataset will have columns like order_id, product_id, customer_id, date, amount, quantity, status, ship_cost, ship_mode, ship_date, remarks. Generate around 10000 rows and store this dataset as csv file orders.csv in volume brij_lakehouse.bronze.data 

"""

In [0]:
# Create the orders view
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders_view AS
SELECT
  id AS order_id,
  (id % 10000) + 1 AS product_id,
  (id % 1000) + 1 AS customer_id,
  date_add('2024-01-01', id % 365) AS date,
  round(50 + (id % 500) * 2.5, 2) AS amount,
  (id % 10) + 1 AS quantity,
  CASE WHEN id % 4 = 0 THEN 'Shipped'
       WHEN id % 4 = 1 THEN 'Pending'
       WHEN id % 4 = 2 THEN 'Cancelled'
       ELSE 'Returned' END AS status,
  round(5 + (id % 20) * 0.5, 2) AS ship_cost,
  CASE WHEN id % 3 = 0 THEN 'Standard'
       WHEN id % 3 = 1 THEN 'Express'
       ELSE 'Overnight' END AS ship_mode,
  date_add(date_add('2024-01-01', id % 365), (id % 7)) AS ship_date,
  concat('Remark_', id) AS remarks
FROM
  (SELECT sequence(1, 10000) AS ids)
  LATERAL VIEW explode(ids) AS id
""")

# Write the view to CSV in the volume
df_orders = spark.table("orders_view")
df_orders.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/brij_lakehouse/bronze/data/orders.csv")

print("Orders data written successfully to /Volumes/brij_lakehouse/bronze/data/orders.csv")

4. Create and load data into brij_lakehouse.bronze.customers table

In [0]:
# Drop existing table if needed
spark.sql("DROP TABLE IF EXISTS brij_lakehouse.bronze.customers_ingest")

# Read CSV and cast types appropriately
df_customers = spark.read.option("header", "true").csv("/Volumes/brij_lakehouse/bronze/data/customers.csv")

# Cast columns to the correct types
from pyspark.sql.functions import col
df_customers_typed = df_customers.select(
    col("customer_id").cast("int"),
    col("name"),
    col("gender"),
    col("address1"),
    col("address2"),
    col("city"),
    col("state"),
    col("country"),
    col("zip"),
    col("phone_1"),
    col("phone_2"),
    col("age").cast("int")
)

# Write to Delta table
df_customers_typed.write.mode("overwrite").saveAsTable("brij_lakehouse.bronze.customers_ingest")

print("bronze.customers_ingest table created and loaded successfully.")

## 5. Create table brij_lakehouse.bronze.customers and load data

In [0]:
# Drop existing table if needed
spark.sql("DROP TABLE IF EXISTS brij_lakehouse.bronze.products_ingest")

# Read CSV and cast types appropriately
df_products = spark.read.option("header", "true").csv("/Volumes/brij_lakehouse/bronze/data/prodcuts.csv")

# Cast columns to the correct types
df_products_typed = df_products.select(
    col("product_id").cast("int"),
    col("name"),
    col("price").cast("double"),
    col("color"),
    col("weight").cast("double"),
    col("vendor_name")
)

# Write to Delta table
df_products_typed.write.mode("overwrite").saveAsTable("brij_lakehouse.bronze.products_ingest")

print("bronze.products_ingest table created and loaded successfully.")

In [0]:
# Drop existing table if needed
spark.sql("DROP TABLE IF EXISTS brij_lakehouse.bronze.orders_ingest")

# Read CSV and cast types appropriately
df_orders = spark.read.option("header", "true").csv(
    "/Volumes/brij_lakehouse/bronze/data/orders.csv"
)

# Cast columns to the correct types
df_orders_typed = df_orders.select(
    col("order_id").cast("int"),
    col("product_id").cast("int"),
    col("customer_id").cast("int"),
    col("date"),
    col("amount").cast("double"),
    col("quantity").cast("int"),
    col("status"),
    col("ship_cost").cast("double"),
    col("ship_mode"),
    col("ship_date"),
    col("remarks"),
)

# Write to Delta table
df_orders_typed.write.mode("overwrite").saveAsTable(
    "brij_lakehouse.bronze.orders_ingest"
)

spark.sql(
    """
MERGE INTO brij_lakehouse.bronze.orders_ingest AS o
USING brij_lakehouse.bronze.products_ingest AS p
ON o.product_id = p.product_id
WHEN MATCHED THEN
  UPDATE SET o.amount = p.price * o.quantity
"""
)

print("bronze.orders_ingest table created and loaded successfully.")

In [0]:
spark.sql(
    """
CREATE OR REPLACE TEMP VIEW orders_view AS
SELECT
  o.*,
  p.price AS unit_price
FROM
  brij_lakehouse.bronze.orders_ingest o
JOIN
  brij_lakehouse.bronze.products_ingest p
ON
  o.product_id = p.product_id
"""
)

spark.sql(
    """
CREATE OR REPLACE TABLE brij_lakehouse.silver.orders AS
SELECT order_id, product_id, customer_id, date as order_date, amount, unit_price, quantity, ship_cost, ship_mode, ship_date, status, remarks
FROM
  orders_view
"""
)

In [0]:
%sql
create or replace table brij_lakehouse.silver.customers as
select
  customer_id,
  name,
  gender,
  address1,
  address2,
  city,
  state,
  case 
    when customer_id % 4 = 0 then 'India'
    when customer_id % 4 = 1 then 'USA'
    when customer_id % 4 = 2 then 'UK'
    else 'Dubai'
  end as country,
  zip,
  phone_1,
  phone_2,
  age,
  case
    when age < 18 then 'adolescence'
    when age < 25 then 'adulthood'
    when age < 50 then 'middle_adulthood'
    else 'late_adulthood'
  end as age_group,
  current_timestamp() as ingest_ts
from
  brij_lakehouse.bronze.customers_ingest;

  select * from brij_lakehouse.silver.customers limit 10;

In [0]:
%sql
create or replace table brij_lakehouse.silver.products as
select
  product_id,
  name,
  case
    when rand() > 0.3 then 'Electronics'
    when rand() > 0.5 then 'Mobile'
    when rand() > 0.8 then 'Baby Care'
    when rand() > 0.9 then 'Beauty'
    when rand() > 0.95 then 'Furniture'
    else 'Other'
  end as category,
  price,
  'USD' as currency,
  color,
  weight,
  current_timestamp() as ingest_ts
from
  brij_lakehouse.bronze.products_ingest;

select * from brij_lakehouse.silver.products limit 10;

In [0]:
# Create dimension tables
spark.sql("""
CREATE OR REPLACE TABLE brij_lakehouse.gold.dim_customer AS
SELECT
  customer_id,
  name,
  gender,
  city,
  state,
  country,
  zip,
  age,
  age_group
FROM brij_lakehouse.silver.customers
""")

spark.sql("""
CREATE OR REPLACE TABLE brij_lakehouse.gold.dim_product AS
SELECT
  product_id,
  name,
  category,
  price,
  currency,
  color,
  weight
FROM brij_lakehouse.silver.products
""")

spark.sql("""
CREATE OR REPLACE TABLE brij_lakehouse.gold.dim_date AS
SELECT DISTINCT
  order_date AS date,
  year(order_date) AS year,
  month(order_date) AS month,
  day(order_date) AS day,
  weekday(order_date) AS weekday
FROM brij_lakehouse.silver.orders
""")

# Create fact table
spark.sql("""
CREATE OR REPLACE TABLE brij_lakehouse.gold.fact_orders AS
SELECT
  order_id,
  product_id,
  customer_id,
  order_date,
  quantity,
  amount,
  unit_price,
  ship_cost,
  ship_mode,
  ship_date,
  status
FROM brij_lakehouse.silver.orders
""")